In [13]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, classification_report

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

heart = pd.read_csv("Heart_Dataset.csv")
ford = pd.read_csv("ford_car_dataset.csv")

le = LabelEncoder()
for col in heart.select_dtypes(include=['object', 'string']).columns:
    heart[col] = le.fit_transform(heart[col])

X_class = heart.drop("HeartDisease", axis=1)
y_class = heart["HeartDisease"]

ford = pd.get_dummies(ford, columns=['transmission', 'fuelType', 'model'], drop_first=True)
X_reg = ford.drop("price", axis=1)
y_reg = ford["price"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler_c = StandardScaler().fit(Xc_train)
scaler_r = StandardScaler().fit(Xr_train)
Xc_train_s = scaler_c.transform(Xc_train)
Xc_test_s = scaler_c.transform(Xc_test)
Xr_train_s = scaler_r.transform(Xr_train)
Xr_test_s = scaler_r.transform(Xr_test)

C:\Users\user\AppData\Local\Temp\ipykernel_26820\3706169695.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in heart.select_dtypes(include='object').columns:


In [ ]:
models_class = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

results_class = []
for name, model in models_class.items():
    model.fit(Xc_train_s, yc_train)
    y_pred = model.predict(Xc_test_s)
    acc = accuracy_score(yc_test, y_pred)
    cm = confusion_matrix(yc_test, y_pred)
    cr = classification_report(yc_test, y_pred)
    print(name)
    print("Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", cr)
    results_class.append([name, acc])

df_class = pd.DataFrame(results_class, columns=["Model", "Accuracy"]).sort_values("Accuracy", ascending=False)
print(df_class)

In [ ]:

models_reg = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(),
    "SVR": SVR(),
    "KNN Regressor": KNeighborsRegressor()
}

results_reg = []
for name, model in models_reg.items():
    model.fit(Xr_train_s, yr_train)
    y_pred = model.predict(Xr_test_s)
    r2 = r2_score(yr_test, y_pred)
    mse = mean_squared_error(yr_test, y_pred)
    print(name)
    print("R2 Score:", r2)
    print("MSE:", mse)
    results_reg.append([name, r2, mse])

df_reg = pd.DataFrame(results_reg, columns=["Model", "R2 Score", "MSE"]).sort_values("R2 Score", ascending=False)
print(df_reg)

In [ ]:
import joblib

best_class_name = df_class.iloc[0]["Model"]
best_reg_name = df_reg.iloc[0]["Model"]

best_class_model = models_class[best_class_name]
best_reg_model = models_reg[best_reg_name]

joblib.dump(best_class_model, "best_heart_model.pkl")
joblib.dump(best_reg_model, "best_ford_model.pkl")
joblib.dump(scaler_c, "scaler_heart.pkl")
joblib.dump(scaler_r, "scaler_ford.pkl")
joblib.dump(list(X_class.columns), "heart_columns.pkl")
joblib.dump(list(X_reg.columns), "ford_columns.pkl")

print("Best Classification Model:", best_class_name)
print("Best Regression Model:", best_reg_name)